# QVerse — Introduction to Quantum Computing & Programming
        ## Week 13: Hybrid Quantum–Classical Workflows

        **Level:** Beginner  
        **Recommended study time:** 2–4 hours  
        **Prerequisites:** Weeks 1–12

        ### Learning objectives
        - Build parameterized quantum circuits.
- Compute expectation values with `StatevectorEstimator`.
- Plot a cost landscape.
- Implement a simple classical optimization loop.

        ---
        **How to use this notebook**

        1. Read the short theory sections.
        2. Make a prediction before running each guided experiment.
        3. Run and modify the code.
        4. Complete every **TODO** exercise.
        5. Finish the reflection section in your own words.

        The goal is not to memorize syntax. The goal is to connect **quantum idea → circuit → result → explanation**.

In [ ]:
# Run this only if your environment does not have the required packages.
# In a terminal, the preferred setup is:
# python -m pip install "qiskit[visualization]>=2.5" matplotlib numpy

# In a fresh Colab notebook you can instead uncomment:
# %pip install "qiskit[visualization]>=2.5" matplotlib numpy -q

## 1. Hybrid workflow

Many near-term algorithms use a loop:

**parameters → quantum circuit → expectation value → classical update → new parameters**

An expectation value of an observable $A$ is written $\langle A\rangle$. For Pauli Z on a qubit:
- `|0>` gives +1;
- `|1>` gives -1;
- superpositions give intermediate values.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

## 2. Parameterized ansatz

In [ ]:
theta = Parameter("θ")

ansatz = QuantumCircuit(1)
ansatz.ry(theta, 0)

observable = SparsePauliOp.from_list([("Z", 1.0)])
estimator = StatevectorEstimator()

ansatz.draw("mpl")

## 3. Evaluate `<Z>` over a parameter sweep

In [ ]:
angles = np.linspace(0, 2*np.pi, 101)
evs = []

for value in angles:
    pub = (ansatz, observable, [value])
    result = estimator.run([pub]).result()[0]
    evs.append(float(np.asarray(result.data.evs)))

plt.plot(angles, evs)
plt.xlabel("θ")
plt.ylabel("<Z>")
plt.title("Expectation value landscape")
plt.show()

## 4. Grid-search minimization

In [ ]:
grid = np.linspace(0, 2*np.pi, 361)
costs = []

for value in grid:
    result = estimator.run([(ansatz, observable, [value])]).result()[0]
    costs.append(float(np.asarray(result.data.evs)))

best_idx = int(np.argmin(costs))
print("Best θ:", grid[best_idx])
print("Minimum <Z>:", costs[best_idx])

The minimum occurs near $\theta=\pi$, where `Ry(pi)|0> = |1>` and therefore $\langle Z\rangle=-1$.

This is a toy version of variational energy minimization: choose an ansatz, evaluate a Hamiltonian expectation value, and let a classical procedure search parameters.

## 5. Tiny one-qubit Hamiltonian

In [ ]:
# H = 0.7 Z + 0.3 X
H = SparsePauliOp.from_list([("Z", 0.7), ("X", 0.3)])

energy = []
for value in grid:
    result = estimator.run([(ansatz, H, [value])]).result()[0]
    energy.append(float(np.asarray(result.data.evs)))

idx = int(np.argmin(energy))
print("Best θ:", grid[idx])
print("Minimum energy:", energy[idx])

plt.plot(grid, energy)
plt.xlabel("θ")
plt.ylabel("<H>")
plt.title("Toy variational energy landscape")
plt.show()

## Core exercises
1. Create an `Ry(theta)` ansatz and evaluate `<Z>` at 0, pi/2, and pi.
2. Plot `<Z>` versus theta from 0 to 2pi.
3. Use grid search to locate the minimum.
4. Replace Z with the toy Hamiltonian `0.7 Z + 0.3 X` and find its best parameter.

In [ ]:
# TODO: Write your solutions here.
# Add extra code cells when useful.

## Optional stretch challenge
Build a two-qubit ansatz with two rotation parameters and one entangling gate. Choose a simple two-qubit Pauli Hamiltonian and visualize or search its two-dimensional cost landscape.

In [ ]:
# OPTIONAL TODO: Attempt the stretch challenge here.

## Weekly reflection
- What makes a workflow hybrid?
- What is the difference between sampling bitstrings and estimating an observable?
- Why does the choice of ansatz matter?

## Submission checklist
- [ ] I made at least one prediction before executing a circuit.
- [ ] All guided examples run.
- [ ] I completed the core exercises.
- [ ] I explained the important output rather than only displaying it.
- [ ] My notebook is readable from top to bottom.